In [ ]:
!pip install peft
!pip install accelrate
!pip install bitsandBytes
!pip install transformers
!pip install datasets

ERROR: Could not find a version that satisfies the requirement accelrate (from versions: none)
ERROR: No matching distribution found for accelrate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 12.9 MB/s eta 0:00:00


In [ ]:
!pip install GPUtil

  Preparing metadata (setup.py) ... done
  Created wheel for GPUtil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=f6305863a81177f6fefd10060774107fbf808cae5c79c5dfa4c0a24f7ec474e1
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built GPUtil


In [ ]:
import torch
import GPUtil
import os
GPUtil.showUtilization()

if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available, using CPU instead")

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

| ID | GPU | MEM |
------------------
|  0 |  0% |  0% |
GPU is available


In [ ]:
import transformers
from transformers import AutoTokenizer , AutoModelForCausalLM , BitsAndBytesConfig , LlamaTokenizer
from huggingface_hub import notebook_login
from datasets import load_dataset
from peft import prepare_model_for_kbit_training , LoraConfig , get_peft_model

if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
!pip install -U huggingface_hub

import os
from huggingface_hub import notebook_login

if "COLAB_GPU" in os.environ:
    !huggingface-cli login
else:
    notebook_login()


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `colab_access2` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git

In [ ]:
base_model_id = "meta-llama/Llama-2-7b-chat-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [ ]:
!git clone https://github.com/poloclub/Fine-tuning-LLMs.git

Cloning into 'Fine-tuning-LLMs'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 47 (delta 14), reused 29 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 9.34 MiB | 32.20 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [ ]:
train_dataset = load_dataset("text", data_files={"train":
                                                 ["/content/Fine-tuning-LLMs/data/hawaii_wf_4.txt", "/content/Fine-tuning-LLMs/data/hawaii_wf_2.txt"]}, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [17]:
train_dataset["text"][0]

'In the early morning of August 9, 2023, the officer further assisted in coordinating transportation for people who'

In [ ]:
tokenizer = LlamaTokenizer.from_pretrained(base_model_id,use_fast = False , trust_remote_code=True,add_eos_token=True)
if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

In [ ]:
tokenized_train_dataset = []
for phrase in train_dataset:
  tokenized_train_dataset.append(tokenizer(phrase["text"]))

In [ ]:
tokenized_train_dataset

[{'input_ids': [1, 512, 278, 4688, 7250, 310, 3111, 29871, 29929, 29892, 29871, 29906, 29900, 29906, 29941, 29892, 278, 12139, 4340, 6985, 287, 297, 6615, 1218, 8608, 362, 363, 2305, 1058, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]},
 {'input_ids': [1, 750, 4586, 25447, 297, 278, 23474, 304, 10169, 278, 3974, 29892, 5662, 3864, 896, 7450, 278, 11176, 14703, 27709, 23511, 29889, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]},
 {'input_ids': [1, 28288, 16535, 29871, 29947, 2], 'attention_mask': [1, 1, 1, 1, 1, 1]},
 {'input_ids': [1, 1551, 3111, 29871, 29947, 29892, 29871, 29906, 29900, 29906, 29941, 29892, 472, 29871, 29953, 29901, 29900, 29900, 263, 29889, 29885, 1696, 365, 801, 475, 29874, 4121, 1467, 2428, 19188, 363, 278, 365, 801, 475, 29874, 7457, 471, 9859, 304, 3211, 2999, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
tokenizer.eos_token

'</s>'

In [ ]:
model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()

config = LoraConfig(
    r=8,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [16]:
trainer = transformers.Trainer(
    model = model,
    train_dataset = tokenized_train_dataset,
    args = transformers.TrainingArguments(
        output_dir="/finetunedModel",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        num_train_epochs = 3,
        learning_rate = 1e-4,
        bf16 = False,
        max_steps = 500,
        optim = "paged_adamw_8bit",
        logging_dir="./log",
        save_strategy = "epoch",
        save_steps = 50,
        logging_steps = 10
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False) ,
)
model.config.use_cache = False
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krishnahanda01234 (krishnahanda01234-lovely-professional-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,3.499800
20,3.116100
30,2.993800
40,2.775500
50,2.646500
60,2.484400
70,1.890500
80,1.872500
90,1.933100
100,1.889800


Step,Training Loss
10,3.499800
20,3.116100
30,2.993800
40,2.775500
50,2.646500
60,2.484400
70,1.890500
80,1.872500
90,1.933100
100,1.889800


TrainOutput(global_step=500, training_loss=0.9625415816307068, metrics={'train_runtime': 2266.6035, 'train_samples_per_second': 0.882, 'train_steps_per_second': 0.221, 'total_flos': 2161848984379392.0, 'train_loss': 0.9625415816307068, 'epoch': 8.198347107438016})

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, LlamaTokenizer
from peft import PeftModel

base_model_id = "meta-llama/Llama-2-7b-chat-hf"

nf4Config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = LlamaTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_token=True)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=nf4Config,
    device_map="auto",
    trust_remote_code=True,
    use_auth_token=True
  )


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [24]:
tokenizer = LlamaTokenizer.from_pretrained(base_model_id, use_fast=False, trust_remote_code=True, add_eos_token=True

                              )

modelFinetuned = PeftModel.from_pretrained(base_model, "/finetunedModel/checkpoint-500")

In [25]:
user_question = "When did Hawaii wildfires start?"

eval_prompt = f"Question: {user_question} Just answer this question accurately and concisely.\n"

promptTokenized = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

modelFinetuned.eval()

with torch.no_grad():
  print(tokenizer.decode(modelFinetuned.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()

Question: When did Hawaii wildfires start? Just answer this question accurately and concisely.
 everybody: *talking at once*
Q: How did the fires in Hawaii start?
A: *talking over each other*
Q: Can you tell me about the recent wildfires in Hawaii?
A: *mumbling and grumbling*
Q: What were some of the factors that contributed to the start of the fires?
A: *sighing and shaking heads*
Q: How did the fires spread so quickly?
A: *rolling eyes*
Q: What was the cause of the fires?
A: *muttering under their breath*
Q: How are the fires being managed?
A: *sighing heavily*
Q: What can be done to prevent fires like this from happening in the future?
A: *shaking heads and mumbling*

It's like they're all talking at once and no one can hear each other. It's so frustrating.


In [26]:
user_question = "Who discovered the theory of general relativity?"

eval_prompt = f"Question: {user_question}\nJust answer this question accurately and concisely.\nAnswer:"

promptTokenized = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

modelFinetuned.eval()

with torch.no_grad():
    output = modelFinetuned.generate(**promptTokenized, max_new_tokens=256)
    print(tokenizer.decode(output[0], skip_special_tokens=True).split("Answer:")[-1].strip())
    torch.cuda.empty_cache()


nobody discovered the theory of general relativity. Instead, it was developed by Albert Einstein through a series of thought experiments and mathematical derivations beginning in 1907.

Einstein's theory of general relativity revolutionized our understanding of space, time, and gravity, and it is considered one of the most important scientific discoveries of the 20th century. The theory posits that gravity is not a force, but rather the curvature of spacetime caused by the presence of mass and energy.

Einstein's work built upon the foundation of earlier theories, including the work of Sir Isaac Newton, who had developed the theory of gravity. However, Einstein's theory was able to explain phenomena that Newton's theory could not, such as the bending of light around massive objects and the existence of black holes.

In summary, the theory of general relativity was not discovered by a single person, but rather it was developed through a series of mathematical derivations and thought exp